___
# <center>Atividade: Duas Variáveis</center>
___

## Aula 06

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * escolher a geometria a partir do par de tipos das duas variáveis;
 * ler uma nuvem de pontos e uma linha de tendência sem exagerar na conclusão;
 * comparar uma numérica entre categorias com boxplot;
 * comparar duas categóricas em contagem e em proporção;
 * preparar a tabela com `.assign()` antes de fazer o gráfico;
 * saber quando a altura da barra é contagem (`geom_bar`) e quando já é uma conta pronta (`geom_col`).

Na aula 5 todo gráfico tinha **uma** variável: quantos acórdãos em cada regime,
como as penas se distribuem. A pergunta era sempre sobre uma coluna sozinha.

Hoje entram duas. A gramática é a mesma, e a única mudança é que o `aes()` passa
a receber dois nomes de coluna em vez de um. Toda a novidade da aula cabe nesta
frase:

> **O par de tipos escolhe a geometria.** Duas numéricas pedem pontos, uma
> numérica com uma categórica pede caixas, duas categóricas pedem barras
> repartidas.

Na segunda hora de hoje é o **Projeto 02**, individual e valendo nota. Ele é a
Gincana do Pipeline no computador: blocos de pandas e de plotnine fora de ordem,
e você organiza. Este notebook tem, de propósito, tudo o que o projeto vai
cobrar e que ainda não apareceu.


___
<div id="indice"></div>

## Índice

- [A tabela de hoje](#dados)

- [De onde a gincana parou](#retomada)
    - [Uma variável numérica sozinha](#histograma)
    - [A coluna que não existe na base](#assign)
    - [Contar não é medir: geom_bar e geom_col](#contarmedir)

- [Uma forma mais curta de escrever](#curta)

- [Duas numéricas: pontos](#numnum)
    - [A linha de tendência](#smooth)
    - [Uma terceira variável na cor](#cor)

- [Numérica e categórica: caixas](#numcat)

- [Duas categóricas: barras repartidas](#catcat)
    - [Contagem ou proporção: position](#position)

- [Preparar a tabela antes do gráfico](#preparar)
    - [Sair do groupby já com a coluna](#asindex)
    - [EXTRA: ordenar as barras com reorder()](#reorder)

- [Que gráfico para que par de variáveis](#escolha)

- [Exercícios](#exercicios)
    - [EXERCÍCIO 1: pena e tamanho da ementa](#ex1)
    - [EXERCÍCIO 2: tráfico e regime](#ex2)
    - [EXERCÍCIO 3: a sua pergunta, com duas variáveis](#ex3)

- [RESUMO](#resumo)


___
<div id="dados"></div>

# A tabela de hoje

A mesma base de apelações criminais da aula 5, e a mesma tabela `penas`.


In [ ]:
# no Colab, rode uma vez:
# %pip install -q plotnine


In [ ]:
import pandas as pd
from plotnine import *

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


In [ ]:
criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

criminal["regime"] = pd.Categorical(
    criminal["regime_inicial"],
    categories=["aberto", "semiaberto", "fechado"],
    ordered=True,
)

penas = (
    criminal
    .dropna(subset=["regime", "pena_anos"])
    .query("pena_anos <= 30")
)

penas.shape


[Volta ao Índice](#indice)


___
<div id="retomada"></div>

# De onde a gincana parou

A gincana da aula 5 não chegou até o fim: ficaram duas rodadas na mesa, e as
duas voltam aqui. Não é revisão: são três ideias que o **Projeto 02** cobra e
que ainda não apareceram escritas.


<div id="histograma"></div>

### Uma variável numérica sozinha

Antes de comparar duas variáveis, vale olhar uma. Uma coluna **categórica** vira
barras, e você já fez isso. Uma coluna **numérica** não tem categorias para
contar: o que se faz é cortar a variável em faixas e contar quantos casos caem
em cada uma. Isso é o histograma.


✔️ **Uso do `geom_histogram()`**

```python
# Sintaxe geral:
geom_histogram(bins=20)
```

Documentação oficial: [geom_histogram()](https://plotnine.org/reference/geom_histogram.html)


In [ ]:
(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=20)
    + labs(x="Pena (anos)", y="Acórdãos")
)


> 🤔 Troque `bins=20` por `bins=5` e depois por `bins=60`. O número de faixas
> muda o que o gráfico deixa ver, e não existe número certo: existe número que
> responde à sua pergunta.

Guarde esta leitura, porque o **boxplot** de hoje é um resumo dela. Onde o
histograma mostra a forma inteira, o boxplot mostra mediana, quartis e pontos
fora, e é isso que permite pôr vários lado a lado.


<div id="assign"></div>

### A coluna que não existe na base

Às vezes a coluna que responde à pergunta simplesmente não está na tabela.
"A capital julga diferente do interior?" precisa de uma coluna que diga capital
ou interior, e a base só tem `comarca`.

`.assign()` cria essa coluna **dentro** do encadeamento, sem quebrar o pipeline
em duas partes. O nome novo vai à esquerda do `=`.


✔️ **Uso do `.assign()`**

```python
# Sintaxe geral:
DataFrame.assign(nome_novo=expressao)
```

Documentação oficial: [.assign()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)


In [ ]:
(
    criminal
    .assign(eh_capital=criminal["comarca"] == "São Paulo")
    .dropna(subset=["pena_anos"])
    .groupby("eh_capital")
    .agg(
        n=("processo", "size"),
        pena_mediana=("pena_anos", "median"),
    )
    .reset_index()
)


> 📌 O `.assign()` precisa vir **antes** do `.groupby()` que usa a coluna nova.
> É a única ordem do pipeline que não pode trocar: a coluna tem que existir para
> poder ser agrupada. Todas as outras operações acima comutam entre si.


<div id="contarmedir"></div>

### Contar não é medir: geom_bar e geom_col

Duas perguntas parecidas, e dois gráficos diferentes:

* *quantos acórdãos em cada regime?* A altura da barra é uma **contagem**, e
  quem conta é a geometria.
* *que proporção dos acórdãos de cada regime é de reincidentes?* A altura é uma
  **conta que você fez**, e a geometria só desenha o número que já está lá.


In [ ]:
# contar: geom_bar() conta as linhas sozinho, e não existe coluna y
(
    ggplot(penas)
    + aes(x="regime")
    + geom_bar()
    + labs(x="Regime inicial", y="Acórdãos")
)


In [ ]:
# medir: primeiro o pandas faz a conta...
resumo_regime = (
    penas
    .groupby("regime", as_index=False, observed=True)
    .agg(prop_reincidencia=("houve_reincidencia", "mean"))
)

resumo_regime


Repare no `"mean"` aplicado a `houve_reincidencia`, que é uma coluna de
verdadeiro e falso. A média de uma coluna assim **é** a proporção de
verdadeiros: verdadeiro conta como 1, falso como 0, e a média de 1 e 0 é a
fração de 1. É o jeito mais curto de calcular proporção em pandas, e o Projeto
02 usa isso nos dois desafios de barras e de linhas.


In [ ]:
# ...e aí o plotnine só desenha a altura que já está na coluna y
(
    ggplot(resumo_regime)
    + aes(x="regime", y="prop_reincidencia")
    + geom_col()
    + labs(x="Regime inicial", y="Proporção com reincidência")
)


> ⚠️ Trocar `geom_col()` por `geom_bar()` aqui não dá erro: dá três barras de
> altura 1, porque cada regime tem uma linha na tabela `resumo_regime` e o
> `geom_bar()` conta linhas. Vale rodar para ver.

**A regra:** a altura já está na tabela? `geom_col()`. A altura é o número de
linhas? `geom_bar()`.


[Volta ao Índice](#indice)


___
<div id="curta"></div>

# Uma forma mais curta de escrever

Na aula 5 escrevemos os dados e a estética em camadas separadas, uma por linha.
Existe uma forma mais curta, em que o `aes()` vai **dentro** do `ggplot()`, como
segundo argumento:


In [ ]:
# as duas células abaixo produzem exatamente o mesmo gráfico

(
    ggplot(penas)
    + aes(x="regime")
    + geom_bar()
)


In [ ]:
(
    ggplot(penas, aes(x="regime"))
    + geom_bar()
)


> 📌 As duas formas convivem, e você vai encontrar as duas em qualquer código
> que buscar na internet. A forma longa é melhor para aprender, porque separa
> as camadas. A curta é a mais comum na prática, e é a que o **Projeto 02**
> usa. Reconhecer as duas é obrigatório; escolher entre elas é gosto.

A regra de sempre continua valendo, e é a mesma da aula 5: **nome de coluna
dentro do `aes()`, valor fixo fora dele**. O que mudou foi só onde o `aes()`
está escrito, não o que ele faz.


[Volta ao Índice](#indice)


___
<div id="numnum"></div>

# Duas numéricas: pontos

Quando as duas variáveis são numéricas, a pergunta é *quando uma cresce, o que
acontece com a outra*, e a geometria é o ponto: um ponto por linha da tabela,
com a posição dada pelas duas colunas.


Agora o `aes()` recebe dois nomes: um para cada eixo.

✔️ **Uso do `geom_point()`**

```python
# Sintaxe geral:
(
    ggplot(tabela, aes(x="coluna_numerica", y="outra_numerica"))
    + geom_point()
)
```

Documentação oficial: [geom_point()](https://plotnine.org/reference/geom_point.html)


In [ ]:
(
    ggplot(penas, aes(x="n_palavras_ementa", y="pena_anos"))
    + geom_point()
    + labs(x="Palavras na ementa", y="Pena (anos)")
)


Os pontos estão empilhados uns sobre os outros na parte de baixo. Quando isso
acontece, `alpha` deixa cada ponto transparente, e a mancha escura passa a
mostrar onde há muitos casos sobrepostos.

`alpha` vai **fora** do `aes()`, porque é um valor fixo: 0 é invisível e 1 é
opaco.


**✍️ Agora você.** Deixe os pontos com 40% de opacidade e um pouco maiores.


In [ ]:
(
    ggplot(penas, aes(x="n_palavras_ementa", y="pena_anos"))
    + geom_point(alpha=0.4, size=2)
    + labs(x="Palavras na ementa", y="Pena (anos)")
)


<div id="smooth"></div>

### A linha de tendência

`geom_smooth(method="lm")` desenha a reta que melhor resume a nuvem, com uma
faixa de incerteza em volta. É uma camada a mais, somada com `+` como qualquer
outra.


In [ ]:
(
    ggplot(penas, aes(x="n_palavras_ementa", y="pena_anos"))
    + geom_point(alpha=0.4)
    + geom_smooth(method="lm", color="#E50505")
    + labs(x="Palavras na ementa", y="Pena (anos)")
)


**A reta é quase plana, e a faixa em volta dela é larga.** Isso é uma resposta,
e uma resposta útil: o tamanho da ementa não diz quase nada sobre a pena.

Duas leituras que o gráfico **não** autoriza:

1. **Reta plana não prova que não há relação.** Prova que não há relação *linear*
   nestes 197 acórdãos, que são os que tinham regime e pena legíveis na ementa.
2. **Reta inclinada não provaria causa.** Se ementas longas viessem com penas
   altas, o mais provável seria que caso grave gera ementa longa *e* pena alta.
   A reta mede associação, e associação não é causa.

> ⚠️ A ordem das camadas importa para o que fica **por cima**: `geom_point()`
> antes de `geom_smooth()` deixa a linha visível sobre os pontos. Trocar a ordem
> não muda os dados, só quem tapa quem.


<div id="cor"></div>

### Uma terceira variável na cor

A mesma nuvem, com `color` dentro do `aes()`: cada regime ganha uma cor, e a
partir daí o `geom_smooth()` desenha **uma reta por grupo**.


In [ ]:
(
    ggplot(penas, aes(x="n_palavras_ementa", y="pena_anos", color="regime"))
    + geom_point(alpha=0.6)
    + geom_smooth(method="lm", se=False)
    + labs(x="Palavras na ementa", y="Pena (anos)", color="Regime inicial")
)


É a mesma ideia do `fill` da aula 5: dentro do `aes()`, a cor virou variável.
E repare no `labs(color=...)`: o `labs()` dá nome a qualquer estética, não só
aos eixos. Sem ele, a legenda sairia escrita `regime`.

> 🤔 Quando o eixo x é tempo, a geometria que liga os pontos é `geom_line()`, e
> `color` dentro do `aes()` desenha uma linha por categoria. É exatamente o que
> você acabou de fazer, trocando `geom_point` por `geom_line`.


[Volta ao Índice](#indice)


___
<div id="numcat"></div>

# Numérica e categórica: caixas

Quando uma variável é numérica e a outra é categórica, a pergunta é *a
distribuição da numérica muda entre as categorias*, e a geometria é o boxplot:
uma caixa por categoria.


✔️ **Uso do `geom_boxplot()`**

```python
# Sintaxe geral:
(
    ggplot(tabela, aes(x="categorica", y="numerica"))
    + geom_boxplot()
)
```

Documentação oficial: [geom_boxplot()](https://plotnine.org/reference/geom_boxplot.html)


In [ ]:
(
    ggplot(penas, aes(x="regime", y="pena_anos"))
    + geom_boxplot(fill="#DCDCDC")
    + labs(x="Regime inicial", y="Pena (anos)")
)


Cada caixa é o resumo da aula 3, desenhado: a linha do meio é a **mediana**, a
caixa vai do primeiro ao terceiro **quartil**, os fios vão até os valores ainda
considerados típicos, e os pontos soltos são os distantes.

A leitura sai direta: a mediana sobe de cerca de 1,4 ano no regime aberto para
3,1 no semiaberto e 5,8 no fechado, e as caixas quase não se sobrepõem. É a
mesma conclusão da tabela da rodada 3 da gincana, agora em uma imagem.

> ⚠️ O boxplot resume, e resumir é esconder. Duas distribuições muito diferentes
> podem ter a mesma caixa. Quando a forma importar, o histograma com facetas da
> aula 5 mostra o que a caixa apagou.


Com muitas categorias, ou com nomes longos, vale deitar: `coord_flip()` de novo,
igual à aula 5.


**✍️ Agora você.** Deite as caixas de `pena_anos` por `classe`.


In [ ]:
(
    ggplot(penas, aes(x="classe", y="pena_anos"))
    + geom_boxplot(fill="#DCDCDC")
    + coord_flip()
    + labs(x="Classe processual", y="Pena (anos)")
)


[Volta ao Índice](#indice)


___
<div id="catcat"></div>

# Duas categóricas: barras repartidas

Quando as duas são categóricas, a pergunta é *a distribuição de uma muda
conforme a outra*, e a geometria é a barra, com a segunda variável no `fill`.

Você já fez isso na rodada 5 da gincana:


In [ ]:
(
    ggplot(penas, aes(x="regime", fill="houve_reincidencia"))
    + geom_bar()
    + labs(x="Regime inicial", y="Acórdãos", fill="Reincidência")
)


E já ouviu o problema dele: **as três barras têm alturas diferentes**, porque há
mais acórdãos em regime fechado. Comparar as fatias de olho engana, porque você
está comparando pedaços de bolos de tamanhos diferentes.


<div id="position"></div>

### Contagem ou proporção: position

O argumento `position` decide como as fatias se arrumam dentro da barra. São
três valores, e cada um responde a uma pergunta diferente.


✔️ **Uso do `geom_bar(position=)`**

```python
# Sintaxe geral:
geom_bar(position="stack")  # empilhado: o padrão, mostra a contagem
geom_bar(position="fill")   # todas as barras com altura 1: proporção
geom_bar(position="dodge")  # lado a lado: contagem, sem empilhar
```

Documentação oficial: [geom_bar(position=)](https://plotnine.org/reference/geom_bar.html)


In [ ]:
(
    ggplot(penas, aes(x="regime", fill="houve_reincidencia"))
    + geom_bar(position="fill")
    + labs(x="Regime inicial", y="Proporção", fill="Reincidência")
)


Agora sim as três barras têm a mesma altura, e a comparação é honesta: a
proporção de acórdãos com reincidência vai de 0,07 no regime aberto para 0,42 no
semiaberto e 0,51 no fechado.

**É a resposta da pergunta que abriu a aula 4**, e é o mesmo número que a rodada
6 da gincana calculou no pandas. A diferença é que aqui o plotnine calculou a
proporção sozinho, dentro da geometria.

> ⚠️ `position="fill"` esconde quanta gente tem em cada barra. Uma barra com 4
> acórdãos e outra com 400 ficam do mesmo tamanho. Quando o tamanho do grupo
> importa, mostre os dois gráficos, ou escreva o `n` no rótulo.


**✍️ Agora você.** Faça as barras lado a lado, em vez de empilhadas.


In [ ]:
(
    ggplot(penas, aes(x="regime", fill="eh_trafico"))
    + geom_bar(position="dodge")
    + labs(x="Regime inicial", y="Acórdãos", fill="Tráfico")
)


[Volta ao Índice](#indice)


___
<div id="preparar"></div>

# Preparar a tabela antes do gráfico

Você já viu a primeira operação desta lista lá em cima, no `.assign()`. Falta
uma que cai no projeto, o atalho para sair do `groupby` já com a coluna, e uma
extra, o jeito de ordenar barras.


<div id="asindex"></div>

### Sair do groupby já com a coluna

Na aula 5 você usou `.reset_index()` depois de todo `groupby`, para trazer a
coluna de agrupamento de volta para dentro da tabela. Existe um atalho:
`as_index=False` dentro do próprio `groupby()` faz a mesma coisa, e poupa uma
peça.


In [ ]:
# as duas células abaixo devolvem exatamente a mesma tabela

(
    criminal
    .groupby("regime_inicial")
    .agg(n=("processo", "size"))
    .reset_index()
)


In [ ]:
(
    criminal
    .groupby("regime_inicial", as_index=False)
    .agg(n=("processo", "size"))
)


> 📌 O `Projeto 02` usa a forma com `as_index=False`. As duas estão certas, e
> reconhecer que fazem a mesma coisa evita que você procure um `.reset_index()`
> que não existe.

Para agrupar por **duas** colunas, passe uma lista. O resultado tem uma linha por
combinação, e é a tabela típica de um gráfico com `color` ou `fill`.


In [ ]:
(
    penas
    .groupby(["regime", "eh_trafico"], as_index=False, observed=True)
    .agg(n=("processo", "size"), pena_mediana=("pena_anos", "median"))
)


<div id="reorder"></div>

### EXTRA: ordenar as barras com reorder()

> 🎁 Esta parte é **extra**: não foi apresentada em aula e o projeto não cobra
> ela. Está aqui porque é a primeira coisa que você vai querer quando fizer um
> gráfico de barras por conta própria.

Ordenar a **tabela** com `.sort_values()` não reordena as **barras**: o plotnine
usa a ordem das categorias, não a ordem das linhas. Quem ordena barra é o
`reorder()`, escrito dentro do `aes()`.

`reorder("categoria", "valor")` põe as categorias em ordem crescente do valor.


In [ ]:
resumo_comarca = (
    penas
    .groupby("comarca", as_index=False)
    .agg(pena_mediana=("pena_anos", "median"), n=("processo", "size"))
    .query("n >= 5")
)

resumo_comarca


**✍️ Agora você.** Ordene as barras da menor para a maior pena mediana, usando `reorder()` no eixo x. Depois `coord_flip()` deita, e a maior fica em cima.


In [ ]:
(
    ggplot(resumo_comarca, aes(x="reorder(comarca, pena_mediana)", y="pena_mediana"))
    + geom_col(fill="#E50505")
    + coord_flip()
    + labs(x="Comarca", y="Pena mediana (anos)")
)


Repare que a altura já estava calculada na coluna `pena_mediana`, então a
geometria é `geom_col()`, e não `geom_bar()`. Essa parte **não** é extra: é a
pegadinha da rodada 6 da gincana, e ela reaparece no projeto.


[Volta ao Índice](#indice)


___
<div id="escolha"></div>

# Que gráfico para que par de variáveis

| as duas variáveis | a pergunta | a geometria |
|---|---|---|
| numérica × numérica | quando uma cresce, o que a outra faz | `geom_point()`, com `geom_smooth()` por cima |
| numérica × numérica, x é tempo | como evolui | `geom_line()` |
| numérica × categórica | a distribuição muda entre as categorias | `geom_boxplot()` |
| numérica × categórica, já resumida | comparar um valor por categoria | `geom_col()` |
| categórica × categórica | a composição muda entre as categorias | `geom_bar(position="fill")` |
| categórica × categórica, contagem | quantos casos em cada combinação | `geom_bar(position="dodge")` |

A terceira variável, quando existe, entra em `color` ou `fill` dentro do
`aes()`, ou em `facet_wrap()` quando o gráfico ficar carregado demais.


[Volta ao Índice](#indice)


___
<div id="exercicios"></div>

# Exercícios


<div id="ex1"></div>

### EXERCÍCIO 1

A pena e o tamanho da ementa, separando por tráfico.

Faça a nuvem de pontos de `pena_anos` contra `n_palavras_ementa`, com a cor
mapeada em `eh_trafico`, e uma reta de tendência por grupo. Depois escreva, em
uma linha, o que o gráfico mostra e o que ele **não** permite concluir.


In [ ]:
(
    ggplot(penas, aes(x="n_palavras_ementa", y="pena_anos", color="eh_trafico"))
    + geom_point(alpha=0.6)
    + geom_smooth(method="lm", se=False)
    + labs(x="Palavras na ementa", y="Pena (anos)", color="Tráfico")
)

# mostra: as duas retas são quase planas, e os acórdãos de tráfico
# aparecem com penas um pouco maiores em quase toda a faixa.
# não permite concluir: nada sobre causa, e nada sobre os acórdãos
# em que a pena não foi lida da ementa, que são mais da metade.


<div id="ex2"></div>

### EXERCÍCIO 2

Tráfico e regime, em proporção.

A proporção de acórdãos de tráfico muda conforme o regime inicial? Faça o
gráfico que responde isso **em proporção**, e não em contagem.


In [ ]:
(
    ggplot(penas, aes(x="regime", fill="eh_trafico"))
    + geom_bar(position="fill")
    + labs(x="Regime inicial", y="Proporção", fill="Tráfico")
)


<div id="ex3"></div>

### EXERCÍCIO 3

A sua pergunta, com duas variáveis.

Escolha **duas** colunas da base e escreva uma pergunta que só se responde
olhando as duas juntas. Depois:

1. diga o tipo de cada uma;
2. escolha a geometria pela tabela da seção anterior;
3. faça o gráfico, com `labs()` preenchido;
4. escreva o que ele mostra e o que ele não permite concluir.

O passo 1 é o que decide os outros três. Se você não souber dizer o tipo das
duas, o gráfico vai sair errado.


In [ ]:
# pergunta: a proporção de reincidência muda conforme a comarca é a
# capital ou não?
# tipos: eh_capital é categórica, houve_reincidencia é categórica

(
    ggplot(
        penas.assign(eh_capital=penas["comarca"] == "São Paulo"),
        aes(x="eh_capital", fill="houve_reincidencia"),
    )
    + geom_bar(position="fill")
    + labs(x="É da capital", y="Proporção", fill="Reincidência")
)

# mostra: as duas barras ficam parecidas, então a comarca ser a capital
# não separa os acórdãos quanto a reincidência.
# não permite concluir: nada sobre o resto do estado agrupado, que
# junta comarcas muito diferentes entre si.


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

Com duas variáveis, a gramática não muda: o que muda é que o `aes()` recebe dois
nomes de coluna, e o par de tipos escolhe a geometria.

| camada | para quê |
|---|---|
| `ggplot(tabela, aes(x=..., y=...))` | a forma curta: dados e estética juntos |
| `+ geom_point(alpha=, size=)` | duas numéricas, um ponto por linha |
| `+ geom_smooth(method="lm", se=False)` | a reta que resume a nuvem |
| `+ geom_line()` | duas numéricas quando o x é tempo |
| `+ geom_boxplot()` | uma numérica comparada entre categorias |
| `+ geom_col()` | a altura já calculada numa coluna |
| `+ geom_bar(position="fill")` | duas categóricas, em proporção |
| `+ geom_bar(position="dodge")` | duas categóricas, em contagem, lado a lado |
| `+ labs(x=, y=, color=, fill=)` | rótulos, inclusive o da legenda |

E três operações de pandas que quase todo gráfico calculado precisa:

| operação | para quê |
|---|---|
| `.assign(nova=expressao)` | criar uma coluna sem quebrar o encadeamento |
| `.groupby(col, as_index=False)` | agrupar já saindo com a coluna na tabela |
| `reorder("cat", "valor")` | ordenar as barras, dentro do `aes()` |


**Três regras que valem sempre:**

1. **O par de tipos escolhe a geometria.** Antes de escrever qualquer coisa,
   diga em voz alta o tipo das duas variáveis.
2. **Ordenar a tabela não ordena as barras.** Quem ordena categoria no gráfico é
   o `reorder()`, dentro do `aes()`.
3. **`position="fill"` responde sobre proporção e esconde o tamanho do grupo.**
   Toda escolha de gráfico decide, junto, o que fica visível e o que some.


[Volta ao Índice](#indice)
